### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "baseline" / "XGBoost_thres0.5_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.02,,,,,
1,0,cf_1,,,,,,,18.9566,,,0.0019,1,True,0.026,0.0123
2,0,cf_2,,,,7,,,18.9745,,,0.1258,2,False,0.026,0.0707
3,0,cf_3,,,,,,,18.9745,2,,0.0365,2,True,0.026,0.007
4,0,cf_4,1,,,,,,18.6402,,,0.1474,2,False,0.026,0.0405
5,0,cf_5,3,,,,,,18.6234,,,0.0652,2,False,0.026,0.0433
6,0,cf_6,,,,6,,,17.9418,,,0.151,2,True,0.026,0.0046
7,0,cf_7,,,6,,,,17.0574,,,0.25,2,True,0.026,0.005
8,0,cf_8,,,6,7,,,18.9745,,,0.2508,3,False,0.026,0.1428
9,0,cf_9,1,2,,,,,18.9745,,,0.2508,3,False,0.026,0.0175


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.02,,,,,
9,0,cf_1,,,,,,,18.9566,,,0.0019,1,True,0.026,0.0123
10,0,cf_3,,,,,,,18.9745,2,,0.0365,2,True,0.026,0.007
11,0,cf_6,,,,6,,,17.9418,,,0.151,2,True,0.026,0.0046
12,0,cf_7,,,6,,,,17.0574,,,0.25,2,True,0.026,0.005
1,1,xin,3,4,1,2,3,0,22.3757,0,1.08,,,,,
13,1,cf_2,,3,,,,,22.3757,4,,0.2011,3,True,0.2497,0.1087
14,1,cf_3,,,2,,1,,22.3757,,,0.1713,3,True,0.2497,0.0851
15,1,cf_5,,,,3,1,,22.3757,,,0.1546,3,True,0.2497,0.0534
16,1,cf_6,,,,,1,,22.3757,1,,0.1475,3,True,0.2497,0.0691


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.02,,,,,
9,0,cf_3,,,,,,,18.9745,2,,0.0365,2,True,0.026,0.007
1,1,xin,3,4,1,2,3,0,22.3757,0,1.08,,,,,
10,1,cf_2,,3,,,,,22.3757,4,,0.2011,3,True,0.2497,0.1087
2,2,xin,5,3,1,1,4,0,22.694,7,1.03,,,,,
11,2,cf_4,,,,7,2,,22.6757,,,0.2088,3,True,0.218,0.0639
3,3,xin,3,3,6,6,2,0,24.3418,1,1.08,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.0007,1,False,0.0653,0.0653
4,4,xin,5,4,2,7,1,0,21.2585,3,1.19,,,,,
13,4,cf_8,1,1,,,,,21.2585,,,0.3618,3,True,0.0811,0.0382


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.0123
Valid: True
Changes:
  bmi: 18.9866 → 18.9566

--- cf_3 ---
Predicted risk: 0.007
Valid: True
Changes:
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 2

--- cf_6 ---
Predicted risk: 0.0046
Valid: True
Changes:
  alcfreq: 4 → 6
  bmi: 18.9866 → 17.9418

--- cf_7 ---
Predicted risk: 0.005
Valid: True
Changes:
  cgtsmok: 3 → 6
  bmi: 18.9866 → 17.0574


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_2 ---
Predicted risk: 0.1087
Valid: True
Changes:
  eatveg: 4 → 3
  dosprt: 0 → 4

--- cf_3 ---
Predicted risk: 0.0851
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 →